# BTC Candle Direction Predictor — Full Walkthrough
**Exchange:** Coinbase (BTC/USD) | **Timeframes:** 5m + 15m (multi-timeframe)

This notebook walks through every step:
1. Data collection
2. Feature engineering (60+ indicators)
3. Target creation
4. Model training (LightGBM + XGBoost + Logistic Regression)
5. Walk-forward validation
6. Confidence filtering & signal analysis
7. Live prediction demo

In [ ]:
# ── INSTALL DEPENDENCIES ──────────────────────────────────────────────────────
# Run this cell once
# !pip install ccxt ta lightgbm xgboost scikit-learn pandas numpy scipy shap joblib requests

In [ ]:
import sys
sys.path.insert(0, 'modules')   # so we can import the modular files

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from sklearn.metrics import (accuracy_score, roc_auc_score, brier_score_loss,
                              calibration_curve, confusion_matrix, ConfusionMatrixDisplay)

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
print('Imports OK')

---
## Step 1 — Data Collection

In [ ]:
from data_collector import load_data, fetch_ohlcv_full

# Load cached data (or set refresh=True to re-download)
df_5m  = load_data('5m',  refresh=False)
df_15m = load_data('15m', refresh=False)

print(f'\n5m  candles : {len(df_5m):,}  ({df_5m.index[0].date()} → {df_5m.index[-1].date()})')
print(f'15m candles : {len(df_15m):,}  ({df_15m.index[0].date()} → {df_15m.index[-1].date()})')
df_5m.tail()

In [ ]:
# Plot BTC close price
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

df_5m['close'].resample('4h').last().plot(ax=ax1, color='#F7931A', linewidth=1)
ax1.set_title('BTC/USD Close Price (4h resampled from 5m data)')
ax1.set_ylabel('Price (USDT)')

df_5m['volume'].resample('4h').sum().plot(ax=ax2, color='#4A90D9', alpha=0.7)
ax2.set_title('Volume')
ax2.set_ylabel('Volume')

plt.tight_layout()
plt.show()

---
## Step 2 — Feature Engineering

In [ ]:
from feature_engineer import full_pipeline, get_feature_columns

# Build 5m dataset with 15m MTF features injected
df = full_pipeline(df_5m, df_15m, inject_15m=True)

fc = get_feature_columns(df)
print(f'\nTotal features : {len(fc)}')
print(f'Dataset rows   : {len(df):,}')
print(f'\nFirst 20 features:\n  {fc[:20]}')

In [ ]:
# Feature correlation heatmap (top 20 by variance)
top_var = df[fc].var().nlargest(20).index
corr    = df[top_var].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(top_var)))
ax.set_yticks(range(len(top_var)))
ax.set_xticklabels(top_var, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(top_var, fontsize=8)
plt.colorbar(im, ax=ax)
ax.set_title('Feature Correlation Matrix (top 20 by variance)')
plt.tight_layout()
plt.show()

---
## Step 3 — Target Variable Analysis

In [ ]:
# Target balance
balance = df['target'].value_counts(normalize=True)
print('Target balance:')
print(f'  UP   (1): {balance.get(1.0, 0)*100:.1f}%')
print(f'  DOWN (0): {balance.get(0.0, 0)*100:.1f}%')
print()

# Rolling UP% over time
fig, ax = plt.subplots(figsize=(14, 4))
df['target'].rolling(500).mean().plot(ax=ax, color='#2ECC71', linewidth=1.5)
ax.axhline(0.5, color='red', linestyle='--', linewidth=1, label='50% baseline')
ax.set_title('Rolling UP% (500 candles) — should hover near 50%')
ax.set_ylabel('P(UP)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature vs target correlation (top predictive features)
corr_target = df[fc + ['target']].corr()['target'].drop('target')
top_pos     = corr_target.nlargest(10)
top_neg     = corr_target.nsmallest(10)
top_corr    = pd.concat([top_pos, top_neg]).sort_values()

fig, ax = plt.subplots(figsize=(10, 8))
colors  = ['#E74C3C' if v < 0 else '#2ECC71' for v in top_corr.values]
top_corr.plot(kind='barh', ax=ax, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 20 Feature Correlations with Target (UP/DOWN)')
ax.set_xlabel('Pearson Correlation')
plt.tight_layout()
plt.show()

---
## Step 4 — Model Training

In [ ]:
from model_trainer import prepare_splits, train_lightgbm, train_xgboost, train_logistic

X_train, X_val, X_test, y_train, y_val, y_test = prepare_splits(df, fc)

In [ ]:
# Train LightGBM (primary model)
lgbm = train_lightgbm(X_train, y_train, X_val, y_val, timeframe='5m')

In [ ]:
# Train XGBoost
xgbm = train_xgboost(X_train, y_train, X_val, y_val)

In [ ]:
# Train Logistic Regression baseline
lr = train_logistic(X_train, y_train)

---
## Step 5 — Validation

In [ ]:
from model_trainer import evaluate, walk_forward_cv

results = {}
for name, model in [('Logistic Regression', lr), ('LightGBM', lgbm), ('XGBoost', xgbm)]:
    results[name] = evaluate(model, X_test, y_test, name=name)

In [ ]:
# Walk-forward cross-validation
cv_df = walk_forward_cv(df, fc, n_splits=5)

In [ ]:
# Calibration curves — how well do probabilities reflect reality?
fig, ax = plt.subplots(figsize=(8, 7))
colors  = ['#3498DB', '#2ECC71', '#E74C3C']

for (name, model), color in zip([('LightGBM',lgbm),('XGBoost',xgbm),('Logistic',lr)], colors):
    proba          = model.predict_proba(X_test)[:, 1]
    frac_pos, mean_pred = calibration_curve(y_test, proba, n_bins=10)
    ax.plot(mean_pred, frac_pos, marker='o', label=name, color=color, linewidth=2)

ax.plot([0,1],[0,1], 'k--', label='Perfect calibration', linewidth=1)
ax.set_xlabel('Mean Predicted Probability', fontsize=12)
ax.set_ylabel('Fraction of Positives (actual UP%)', fontsize=12)
ax.set_title('Calibration Curves — Are probabilities reliable?', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

In [ ]:
# Confusion matrix — LightGBM
lgbm_pred = (lgbm.predict_proba(X_test)[:,1] > 0.5).astype(int)
cm = confusion_matrix(y_test, lgbm_pred)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=['DOWN','UP']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('LightGBM Confusion Matrix (Test Set)')
plt.tight_layout()
plt.show()

---
## Step 6 — SHAP Feature Importance

In [ ]:
try:
    import shap
    sample    = X_train.sample(min(1000, len(X_train)), random_state=42)
    explainer = shap.TreeExplainer(lgbm)
    vals      = explainer.shap_values(sample)
    if isinstance(vals, list):
        vals = vals[1]  # P(UP)
    shap.summary_plot(vals, sample, max_display=20, show=True)
except ImportError:
    print('Install shap: pip install shap')
    # Fallback: LightGBM native importance
    imp = pd.Series(lgbm.feature_importances_, index=fc).sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(10,8))
    imp.head(20).plot(kind='barh', ax=ax, color='#3498DB')
    ax.set_title('LightGBM Feature Importance (top 20)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

---
## Step 7 — Confidence Filtering Analysis

In [ ]:
proba = lgbm.predict_proba(X_test)[:, 1]
conf  = np.maximum(proba, 1 - proba)

thresholds = np.arange(0.50, 0.76, 0.01)
rows = []
for t in thresholds:
    mask = conf >= t
    n    = mask.sum()
    if n < 10: continue
    acc = accuracy_score(y_test[mask], (proba[mask] > 0.5).astype(int))
    rows.append({'threshold': t, 'accuracy': acc,
                 'signals': n, 'coverage_pct': n/len(y_test)*100})

tdf = pd.DataFrame(rows)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(tdf['threshold'], tdf['accuracy'], 'g-o', linewidth=2, markersize=5)
ax1.axhline(0.5, color='red', linestyle='--', label='Random baseline')
ax1.set_xlabel('Confidence Threshold'); ax1.set_ylabel('Accuracy')
ax1.set_title('Accuracy vs Confidence Threshold')
ax1.legend(); ax1.set_ylim(0.45, 0.70)

ax2.plot(tdf['threshold'], tdf['coverage_pct'], 'b-o', linewidth=2, markersize=5)
ax2.set_xlabel('Confidence Threshold'); ax2.set_ylabel('% Candles with Signal')
ax2.set_title('Signal Coverage vs Confidence Threshold')

plt.suptitle('Trade-off: Higher confidence = better accuracy but fewer signals', fontsize=12)
plt.tight_layout()
plt.show()

print(tdf[tdf['threshold'].isin([0.55, 0.60, 0.65, 0.70])].to_string(index=False))

---
## Step 8 — Save Models & Test Live Prediction

In [ ]:
from model_trainer import save_models

models = {'logistic': lr, 'lightgbm': lgbm, 'xgboost': xgbm}
ts     = save_models(models, fc, timeframe='5m')
print(f'Models saved with timestamp: {ts}')

In [ ]:
# ── LIVE PREDICTION TEST ──────────────────────────────────────────────────────
# This fetches real current data from Binance and makes a live prediction
from live_predictor import predict_next_candle

result = predict_next_candle(
    timeframe    = '5m',
    models       = models,
    feature_cols = fc,
    threshold    = 0.60,
)

print(f"\nSignal: {result.get('emoji')} {result.get('signal')}")
print(f"P(UP): {result.get('prob_up'):.4f}  P(DOWN): {result.get('prob_down'):.4f}")
print(f"Confidence: {result.get('confidence'):.4f}")

In [ ]:
# To run continuous live predictions:
#
# from live_predictor import run_live
# run_live(timeframes=['5m', '15m'], threshold=0.60)
#
# Or from terminal:
# python modules/live_predictor.py live

print('Live loop ready — uncomment above or run from terminal')